<a href="https://colab.research.google.com/github/SAO-P/Basic-Deep-Learning-2026-Spring/blob/main/lecture02_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 第2回講義 宿題

---
# Lecture 2: Homework


## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

---
## Task
Based on what we learned in this lesson, let's classify the fashion version of MNIST (Fashion MNIST, number of classes 10) using softmax regression.

For more information about Fashion MNIST, please refer to the link below.

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist


### 目標値 (Target)
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **隠れ層のない単層のモデルとして実装してください.**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

---
### Rules
- Training data is given by `x_train`, `y_train`, and test data is given by `x_test`.
- Please express the predicted label as a class label from 0 to 9 instead of one_hot expression.
- **Do not use training data other than** `x_train, y_train` **specified in the cell below.**
- **Please implement as a single layer model without hidden layers.**
- **Please implement the algorithm part of softmax regression only with numpy** (do not use sklearn, tensorflow, etc.).
- There is no problem using sklearn functions (for example, `sklearn.model_selection.train_test_split`) in the data preprocessing part.


### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルをcsv形式で保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを　ファイル＞ダウンロード＞.pyをダウンロード　から保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコード全体をコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

---
### Submission
- Please submit two files.
  1. Save the predicted labels for the test data (`x_test`) in csv format, select "2nd Machine Learning Basics" from the Homework tab of Omnicampus and submit.
  2. Save the corresponding python code from File > Download > Download .py, and select "2nd Machine Learning Fundamentals (code)" from the Omnicampus Homework tab and submit it. Rather than submitting the python file itself, please copy and paste the entire code into the "submission content" section.
      
- Please note that the score will be 1, and 2 will be used to check the code (excellent students may have their code made public). If you change the content of the code, **Please resubmit both 1 and 2**.


### 評価方法
- 予測ラベルの`y_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

---
### Evaluation
- Evaluate the accuracy of the predicted label against `y_test`.
- Immediate grading and updating the Leader Board (grading schedule will be announced separately).
- The score at the deadline will be the final evaluation.


### ドライブのマウント

---
### Mount Google drive


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# Specify working directory
work_dir = 'drive/MyDrive/DL STUDY/MATSUO DL/DL 2026 SPRING'

### データの読み込み（このセルは修正しないでください）

---
### Load Data (Do not modify this cell)


In [7]:
import os
import sys

import numpy as np
import pandas as pd

sys.modules['tensorflow'] = None

def load_fashionmnist():
    # learning data
    x_train = np.load(work_dir + '/Lecture02/data/x_train.npy')
    y_train = np.load(work_dir + '/Lecture02/data/y_train.npy')

    # test data
    x_test = np.load(work_dir + '/Lecture02/data/x_test.npy')

    x_train = x_train.reshape(-1, 784).astype('float32') / 255
    y_train = np.eye(10)[y_train.astype('int32')]
    x_test = x_test.reshape(-1, 784).astype('float32') / 255

    return x_train, y_train, x_test

### ソフトマックス回帰の実装

---
### Implementing softmax regression


In [20]:
x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def softmax(x):
    x -= x.max(axis=1, keepdims=True)
    x_exp = np.exp(x)

    return x_exp / x_exp.sum(axis=1, keepdims=True)

# weight
W = np.random.uniform(low=-0.08, high=0.08, size=(784, 10)).astype('float32')  # Weight: (784, 10)
b = np.zeros(shape=(10,)).astype('float32')

# Split into training data and validation data
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.1)

def train(x, t, eps=1.0):
    global W, b
    batch_size = x.shape[0]

    # Forward: @ 연산자 사용 (행렬 곱)
    y = softmax(x @ W + b)  #y_hat

    # Gradient
    delta = y - t
    dW = (x.T @ delta) / batch_size
    db = np.sum(delta, axis=0) / batch_size

    # Update
    W -= eps * dW
    b -= eps * db

    # Loss (BCE)
    loss = -np.mean(np.sum(t * np.log(y + 1e-7), axis=1))
    return loss

def valid(x, t):
    y = softmax(x @ W + b)
    loss = (- t * np.log(y + 1e-7)).sum(axis=1).mean()

    return loss, y

#training
batch_size = 128
for epoch in range(150):

    if epoch < 40:
        lr = 0.2
    elif epoch < 80:
        lr = 0.1
    elif epoch < 120:
        lr = 0.05
    else:
        lr = 0.01

    # 데이터를 매 에포크마다 mix
    perm = np.random.permutation(len(x_train))
    x_shuffled = x_train[perm]
    y_shuffled = y_train[perm]

    for i in range(0, len(x_train), batch_size):
        x_batch = x_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]


        train(x_batch, y_batch, eps=lr)  #eps=lr

    v_loss, v_y_pred = valid(x_valid, y_valid)


    # 10回ごと出力
    if (epoch + 1) % 10 == 0 or epoch == 0:
        # Accuracy
        v_acc = accuracy_score(y_valid.argmax(axis=1), v_y_pred.argmax(axis=1))

        print('EPOCH: {}, Valid Loss: {:.3f}, Valid Accuracy: {:.3f}'.format(
            epoch + 1,
            v_loss,
            v_acc
        ))
y_pred = softmax(x_test @ W + b)

submission = pd.Series(y_pred.argmax(axis=1), name='label')
submission.to_csv(work_dir + '/Lecture02/submission_pred_02.csv', header=True, index_label='id')

EPOCH: 1, Valid Loss: 0.514, Valid Accuracy: 0.827
EPOCH: 10, Valid Loss: 0.454, Valid Accuracy: 0.841
EPOCH: 20, Valid Loss: 0.418, Valid Accuracy: 0.854
EPOCH: 30, Valid Loss: 0.430, Valid Accuracy: 0.853
EPOCH: 40, Valid Loss: 0.423, Valid Accuracy: 0.857
EPOCH: 50, Valid Loss: 0.424, Valid Accuracy: 0.853
EPOCH: 60, Valid Loss: 0.417, Valid Accuracy: 0.859
EPOCH: 70, Valid Loss: 0.426, Valid Accuracy: 0.852
EPOCH: 80, Valid Loss: 0.417, Valid Accuracy: 0.856
EPOCH: 90, Valid Loss: 0.415, Valid Accuracy: 0.858
EPOCH: 100, Valid Loss: 0.416, Valid Accuracy: 0.858
EPOCH: 110, Valid Loss: 0.416, Valid Accuracy: 0.857
EPOCH: 120, Valid Loss: 0.413, Valid Accuracy: 0.858
EPOCH: 130, Valid Loss: 0.412, Valid Accuracy: 0.860
EPOCH: 140, Valid Loss: 0.412, Valid Accuracy: 0.860
EPOCH: 150, Valid Loss: 0.412, Valid Accuracy: 0.860
